# Dalitz driver: $\eta \to \mu^+\mu^-\gamma$

The Dalitz counterpart of `driver_gauss.py`, run as a notebook so the
intermediate diagnostics stay inspectable:

    read ROOT -> select -> best candidate -> Dalitz variables
              -> global fit_gauss -> per-Dalitz-bin fits -> yield maps

Everything that already exists in the framework is used as-is: `cut_utils` for
reading/selection/best-candidate, `fit_gauss` + `fit_utils` for the fits,
`plot_gauss` for the mass-fit figure, `plot_style`/`plot_utils` for house
style and saving. What is new here is Dalitz-specific: computing
$m^2_{\gamma\mu^+}$ / $m^2_{\mu^+\mu^-}$ from the best candidate's daughters,
fitting the mass spectrum in each Dalitz bin, and the resulting yield maps.

This replaces `fit_dalitz.ipynb`, which predates most of that framework. One
deliberate behavioural difference: the old notebook applied **no** TOS or
`pnn_mu` cut, so its per-bin yields will not reproduce bin for bin.

In [ ]:
import sys
import os
import json

import numpy as np
import awkward as ak
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# Ensure the project modules are importable regardless of working directory.
# Notebooks don't define __file__, so fall back to cwd (VS Code/Jupyter run
# notebooks with cwd set to the notebook's own directory).
try:
    _here = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _here = os.getcwd()
sys.path.insert(0, os.path.join(_here, '..', 'utils'))
sys.path.insert(0, os.path.join(_here, '..', 'fit'))
sys.path.insert(0, os.path.join(_here, '..', 'plot'))

import cut_utils
import fit_utils
import plot_utils
import plot_style
from fit_gauss import fit_gauss
from plot_gauss import plot_gauss

## Configuration

The notebook equivalent of `driver_gauss.py`'s argparse block: argparse doesn't
work in Jupyter, so everything the driver would take on the command line is an
inline variable here.

In [ ]:
INFILE = '../../ntuple/lhcb/eta2mumugamma/20260608/combined_files.root'
OUTDIR = '../../out/eta2mumugamma/turbo/20260608/'
MAX_EVENTS = None      # cap on events read, for quick iteration; None = all

# Branches, matching driver_gauss.py. The prt_px/py/pz/e daughter kinematics are
# the Dalitz-specific addition -- a plain mass fit only ever needs the tag mass.
MASS_BRANCH     = 'tag_dtf_m'       # the quantity being fit
METRIC_BRANCH   = 'tag_dtf_chi2'    # ranks candidates within an event
DAUGHTER_FIELDS = ['prt_pid', 'prt_pnn_mu', 'prt_px', 'prt_py', 'prt_pz', 'prt_e']
BRANCHES = ([MASS_BRANCH, METRIC_BRANCH, 'prt_idx_mom']
            + DAUGHTER_FIELDS + cut_utils.TOS_TRIGGERS)

# Selection, same as driver_gauss.py. Set USE_TOS=False / PNN_MU_MIN=None to
# reproduce the old fit_dalitz.ipynb, which applied neither cut.
USE_TOS    = True
PNN_MU_MIN = 0.95

# Mass fit window. The global fit, every per-bin fit, and the per-bin plots at
# the bottom MUST share these three numbers -- otherwise the histogram being
# reviewed is not the histogram that was fit.
XMIN, XMAX, MASS_NBINS = 400.0, 696.0, 80
MEAN_INIT, SIGMA_INIT = 547.9, 10.0

# Dalitz binning. DALITZ_NBINS is the TOTAL number of Dalitz bins, so the grid is
# sqrt(DALITZ_NBINS) per dimension. It has nothing to do with MASS_NBINS above,
# and the two must never be passed to each other.
DALITZ_NBINS = 400
M12_RANGE = (0.0, 0.21)
M23_RANGE = (0.0, 0.31)
MIN_CANDIDATES = 10    # fewer than this in a Dalitz bin -> don't attempt a fit

# Physical constraints for the Dalitz plot [GeV^2]
M_MU  = 0.105658
M_ETA = 0.547862
M12_PHYS = (M_MU**2, (M_ETA - M_MU)**2)
M23_PHYS = ((2 * M_MU)**2, M_ETA**2)

# Every figure below is both shown inline and written to OUTDIR. The old notebook
# declared results/plot filenames and then never used them.
os.makedirs(OUTDIR, exist_ok=True)

## Read and select

Straight out of `driver_gauss.py`: read the branches, drop events that can't be
interpreted, build a per-candidate selection mask, then reduce each event to its
single best candidate by smallest `tag_dtf_chi2`.

In [ ]:
tree = cut_utils.read_branches(INFILE, BRANCHES, entry_stop=MAX_EVENTS)
tree = cut_utils.drop_malformed(tree, DAUGHTER_FIELDS, cut_utils.TOS_TRIGGERS,
                                tag_ref=MASS_BRANCH)

n_multi = int(ak.sum(ak.num(tree[MASS_BRANCH]) > 1))
print(f'[INFO] {n_multi} / {len(tree)} events have >1 candidate')

In [ ]:
# selection_mask returns a per-candidate boolean mask shaped like MASS_BRANCH.
# Note we never index the whole tree with it -- that would silently truncate the
# prt_* fields.
keep = cut_utils.selection_mask(tree, use_tos=USE_TOS, pnn_mu_min=PNN_MU_MIN,
                                tag_ref=MASS_BRANCH)

# Dalitz-specific requirement on top of the standard cuts: m12/m23 are only
# defined for a candidate carrying exactly one gamma, one mu+ and one mu-. The
# old notebook checked this inline while looping (skipping candidates where any
# of the three came back None). As a per-candidate mask it composes with the rest
# of the selection with a plain `&` and -- importantly -- is applied BEFORE the
# best-candidate argmin, so a malformed candidate can never win an event and
# displace a well-formed one.
#
# "How many daughters of this PID does the candidate have?" is a daughter ->
# candidate question, so it is a reduce() along prt_idx_mom. prt_pid stays in its
# flat ROOT layout; nothing is nested.
def has_exactly_one(pid_code):
    """Per-candidate mask: the candidate has exactly one daughter of this PID."""
    return cut_utils.reduce(tree, tree['prt_pid'] == pid_code, 'prt_idx_mom',
                            how='sum', target_ref=MASS_BRANCH) == 1


dalitz_ok = (has_exactly_one(22)      # gamma
             & has_exactly_one(-13)   # mu+
             & has_exactly_one(13))   # mu-
print(f'[INFO] well-formed (gamma, mu+, mu-) candidates: '
      f'{int(ak.sum(dalitz_ok))} / {int(ak.sum(ak.num(dalitz_ok)))}')

keep = keep & dalitz_ok
n_candidates = int(ak.sum(ak.num(keep)))
n_selected = int(ak.sum(keep))

In [ ]:
# Reduce to one candidate per event. `keep` goes in as the eligibility mask, so
# each event keeps the best candidate that actually SURVIVED the cuts -- ANDing a
# winner mask with `keep` afterwards would instead drop any event whose globally
# best candidate had been cut.
#
# What comes back is a mask on the same candidate axis as `keep`, which is what
# makes the rest of this notebook simple: it indexes any per-candidate branch
# directly, needs no index round-trip, and introduces no option type (so no
# drop_none, and ak.to_numpy gives a plain ndarray).
best = cut_utils.best_candidate_mask(tree[METRIC_BRANCH], keep, method='min')
masses = ak.to_numpy(ak.flatten(tree[MASS_BRANCH][best]))

applied = ', '.join((['TOS'] if USE_TOS else [])
                    + ([f'pnn_mu>{PNN_MU_MIN}'] if PNN_MU_MIN is not None else [])
                    + ['well-formed daughters']) or 'none'
print(f'[INFO] cutflow: {n_candidates} candidates -> cuts [{applied}] -> '
      f'{n_selected} candidates -> {len(masses)} events with a best candidate')

## Dalitz variables

$m^2_{\gamma\mu^+}$ and $m^2_{\mu^+\mu^-}$ for each event's best candidate.
This is the piece with no framework equivalent: `cut_utils` reduces to a best
candidate and hands back a tag-level quantity, whereas the Dalitz plot needs
that candidate's individual daughter four-vectors.

In [ ]:
GEV = 1000.0   # branches are in MeV; Dalitz variables are conventionally GeV^2

# Cross from the candidate level down to the daughter level. The file keeps
# daughters in their own flat per-event list, wired to candidates by prt_idx_mom;
# follow() travels that link, so `dmask` is True exactly for the daughters of the
# candidates `best` selected.
#
# This is the same follow()/reduce() pair used for the selection above, just in
# the other direction -- and it is why nothing here is ever nested into
# [event][candidate][daughter]. That matters beyond tidiness: the MC generator
# branches (mc_*) are a separate collection several candidates can point at
# simultaneously, so they cannot be nested under candidates at all.
dmask = cut_utils.follow(tree, best, 'prt_idx_mom')


def four_vector(pid_code):
    """(px, py, pz, e) in GeV for the daughter of this PID in each event's best
    candidate.

    The selection guarantees exactly one match per selected candidate, so
    flattening gives one value per event, aligned with `masses`.
    """
    match = dmask & (tree['prt_pid'] == pid_code)
    return tuple(ak.to_numpy(ak.flatten(tree[field][match])) / GEV
                 for field in ('prt_px', 'prt_py', 'prt_pz', 'prt_e'))


gamma, mu_plus, mu_minus = four_vector(22), four_vector(-13), four_vector(13)
m12s = cut_utils.invariant_mass2(*gamma, *mu_plus)      # m^2(gamma, mu+)
m23s = cut_utils.invariant_mass2(*mu_plus, *mu_minus)   # m^2(mu+, mu-)

# Same positivity requirement the old notebook applied: a handful of candidates
# reconstruct to a negative m^2 and cannot sit on a Dalitz plot. Masking all
# three arrays together is what keeps them the same length by construction.
physical = (m12s > 0) & (m23s > 0)
n_unphysical = int((~physical).sum())
m12s, m23s, masses = m12s[physical], m23s[physical], masses[physical]
print(f'[INFO] dropped {n_unphysical} events with m12^2 <= 0 or m23^2 <= 0')

In [ ]:
# Check ranges and shapes. m12/m23 ranges are the tell that candidate selection
# still behaves: with the old notebook's (no-cut) selection they came out as
# m12: [0.0119, 0.9969], m23: [0.0447, 1.1849]. Adding TOS + pnn_mu trims the
# sample, so expect these to move -- what should not happen is the ranges
# leaving the kinematically allowed region.
print('--- Dalitz plot statistics ---')
print(f'm12 range:  [{m12s.min():.4f}, {m12s.max():.4f}]  '
      f'(physical: [{M12_PHYS[0]:.4f}, {M12_PHYS[1]:.4f}])')
print(f'm23 range:  [{m23s.min():.4f}, {m23s.max():.4f}]  '
      f'(physical: [{M23_PHYS[0]:.4f}, {M23_PHYS[1]:.4f}])')
print(f'mass range: [{masses.min():.4f}, {masses.max():.4f}]')

print('--- Shape check ---')
print(f'm12s: {m12s.shape}   m23s: {m23s.shape}   masses: {masses.shape}')

## Raw Dalitz distributions

Before any fitting: what the candidates look like on the Dalitz plane, over the
full reconstructed range and zoomed into the physical region.

In [ ]:
def draw_dalitz_axes(ax):
    """Physical-boundary lines and axis labels shared by every Dalitz figure."""
    ax.axvline(M12_PHYS[0], color='red', ls='--', lw=1,
               label='physical signal window')
    ax.axvline(M12_PHYS[1], color='red', ls='--', lw=1)
    ax.axhline(M23_PHYS[0], color='red', ls='--', lw=1)
    ax.axhline(M23_PHYS[1], color='red', ls='--', lw=1)
    ax.legend(loc='upper right', fontsize=8)
    ax.set_xlabel(r'$m^2_{\gamma,\mu^+}$ [GeV$^2$]')
    ax.set_ylabel(r'$m^2_{\mu^+,\mu^-}$ [GeV$^2$]')


def plot_dalitz_hist(m12s, m23s, hist_range, title, output, bins=100):
    """Raw 2D histogram of the Dalitz variables (no fitting involved)."""
    fig, ax = plt.subplots(figsize=(8, 6))
    *_, mesh = ax.hist2d(m12s, m23s, bins=bins, cmap='viridis', range=hist_range)
    fig.colorbar(mesh, ax=ax, label='Counts')
    draw_dalitz_axes(ax)
    ax.set_title(title)
    plot_style.save_plot(fig, output)
    return fig


def plot_dalitz_map(values, m12_edges, m23_edges, cbar_label, title, output):
    """Per-bin quantity (e.g. a fitted yield) drawn on the Dalitz grid."""
    fig, ax = plt.subplots(figsize=(7, 6))
    mesh = ax.pcolormesh(m12_edges, m23_edges, values.T, cmap='viridis',
                         shading='auto')
    fig.colorbar(mesh, ax=ax, label=cbar_label)
    draw_dalitz_axes(ax)
    ax.set_title(title)
    plot_style.save_plot(fig, output)
    return fig


def dalitz_font():
    """House font for the Dalitz figures.

    plot_fit_data() sets font.size globally to 16 for the mass-fit plots, so
    this is re-applied whenever a Dalitz figure follows one of those. The old
    notebook reached for plt.rcdefaults(), which also throws away the house
    font.
    """
    plot_utils.apply_font('sans')
    plt.rcParams['font.size'] = 11

In [ ]:
dalitz_font()
plot_dalitz_hist(m12s, m23s, [[0, 1.0], [0, 1.2]],
                 r'$\eta \to \mu\mu\gamma$ Dalitz plot (all candidates)',
                 os.path.join(OUTDIR, 'dalitz_raw.png'))
plot_dalitz_hist(m12s, m23s, [[0, 0.22], [0, 0.35]],
                 r'$\eta \to \mu\mu\gamma$ Dalitz plot (physical region)',
                 os.path.join(OUTDIR, 'dalitz_raw_zoom.png'))

## Global mass fit

`fit_gauss()` on the whole sample. Reading, selection and best-candidate
reduction all happened upstream, so this global fit runs on exactly the sample
the per-bin fits will be carved out of, and the mean/sigma/c0/c1 it produces
seed those bins consistently.

In [ ]:
result = fit_gauss(masses, xmin=XMIN, xmax=XMAX, nbins=MASS_NBINS,
                   mean_init=MEAN_INIT, sigma_init=SIGMA_INIT)

# Where the numbers came from -- the fit itself only ever saw an array.
result['meta']['input_file'] = INFILE
result['meta']['selection'] = {
    'tos': USE_TOS,
    'pnn_mu_min': PNN_MU_MIN,
    'best_candidate': f'min {METRIC_BRANCH}',
    'dalitz_daughters': 'exactly one each of gamma, mu+, mu-',
}
# Everything the selection did, so the numbers behind this fit can be audited
# from the results file alone rather than from notebook scrollback.
result['selection_report'] = cut_utils.report()
result['selection_report']['best_candidate'] = {
    'metric': METRIC_BRANCH, 'method': 'min',
    'events_with_best_candidate': len(masses),
}
result['selection_report']['branches_read'] = BRANCHES

In [ ]:
# plot_gauss layers its DEFAULT_CONFIG (house font, stats box, pull panel) under
# these overrides -- the same call driver_gauss.py makes.
plot_gauss(result, {'output': os.path.join(OUTDIR, 'global_gauss_fit.png'),
                    'title': 'Eta Mass (global fit)'})

In [ ]:
# Seed the per-bin fits from the global fit.
mean_init  = result['fit']['parameters']['mean']['value']
sigma_init = result['fit']['parameters']['sigma']['value']
c0_init    = result['fit']['parameters']['c0']['value']
c1_init    = result['fit']['parameters']['c1']['value']
print(f'[INFO] per-bin seeds: mean={mean_init:.3f}  sigma={sigma_init:.3f}  '
      f'c0={c0_init:.3f}  c1={c1_init:.3f}')

## Per-Dalitz-bin fits

`fit_gauss()` fits one array and returns a JSON-shaped dict; that is the right
shape for a single global fit, but a few hundred per-bin fits only need the
Minuit object. So these drop one level and call `fit_utils` directly — the same
`make_gauss_cost` / `run_fit` pair `fit_gauss()` itself uses, with the same
model, so the global fit and the per-bin fits are one fit applied to different
subsets.

In [ ]:
# The mass histogram every per-bin fit and every per-bin plot uses. Defined once
# so the two provably agree.
MASS_EDGES = np.linspace(XMIN, XMAX, MASS_NBINS + 1)
MASS_CENTERS = 0.5 * (MASS_EDGES[:-1] + MASS_EDGES[1:])
MASS_BIN_WIDTH = (XMAX - XMIN) / MASS_NBINS


def dalitz_edges(nbins=DALITZ_NBINS, m12_range=M12_RANGE, m23_range=M23_RANGE):
    """Bin edges for the Dalitz grid, plus the number of bins per dimension.

    `nbins` is the TOTAL bin count, so the grid is sqrt(nbins) per dimension.
    """
    n = int(np.sqrt(nbins))
    return np.linspace(*m12_range, n + 1), np.linspace(*m23_range, n + 1), n


def dalitz_bin_mask(m12s, m23s, m12_edges, m23_edges, i, j):
    """Half-open [lo, hi) mask selecting one Dalitz bin.

    Both the fits and the per-bin plots go through this, so a page always shows
    the candidates its own fit saw.
    """
    return ((m12s >= m12_edges[i]) & (m12s < m12_edges[i + 1]) &
            (m23s >= m23_edges[j]) & (m23s < m23_edges[j + 1]))


def fit_gauss_per_bin(masses, mean_init, sigma_init, c0_init, c1_init):
    """Gaussian + Chebyshev fit to one Dalitz bin's mass distribution.

    Binned across [XMIN, XMAX] into MASS_NBINS mass bins -- the notebook-level
    mass binning, never the Dalitz bin count. Slicing the window into a few
    hundred bins instead buries each per-bin fit in Poisson noise.

    Only the signal shape is fixed from the global fit. The signal may vary a
    little across the Dalitz plot, but the background shape is expected to vary
    more, so c0/c1 must float: fixing them would pin every bin's background to
    one global curve with only its normalization (n_bkg = n_cand - n_sig) free,
    which is the opposite of the intent.
    """
    counts, edges = np.histogram(masses, bins=MASS_EDGES)
    counts = counts.astype(float)   # ensure counts are float for fitting
    n_cand = counts.sum()

    cost = fit_utils.make_gauss_cost(counts, edges, XMIN, XMAX)
    init = dict(mean=mean_init, sigma=sigma_init, n_sig=n_cand * 0.5,
                c0=c0_init, c1=c1_init)
    limits = dict(mean=(XMIN, XMAX), sigma=(1.0, 50.0), n_sig=(0.0, n_cand),
                  c0=(-1.0, 1.0), c1=(-1.4, 1.4))
    # m == fit result object holding parameters, errors and goodness-of-fit.
    m = fit_utils.run_fit(cost, init, limits, fixed=['mean', 'sigma'])
    return m, n_cand


def bin_prediction(fit_result, n_cand, counts):
    """Expected counts, pulls and chi2/ndf for one Dalitz bin's fit.

    Mirrors what fit_gauss() computes for the global fit, and uses the same
    fit_utils CDFs the cost function used -- so what gets drawn and reported is
    the fit, not a re-derivation of it.
    """
    p = fit_result.values
    n_sig = float(p['n_sig'])
    # n_bkg is not a fit parameter: n_cand is fixed going in, and n_sig + n_bkg
    # = n_cand by unitarity.
    n_bkg = float(n_cand) - n_sig

    bin_sig = np.diff(fit_utils.gauss_cdf(MASS_EDGES, p['mean'], p['sigma'],
                                          XMIN, XMAX)) * n_sig
    bin_bkg = np.diff(fit_utils.cheb_cdf(MASS_EDGES, XMIN, XMAX,
                                         p['c0'], p['c1'])) * n_bkg
    bin_fit = bin_sig + bin_bkg

    # Pearson pulls and chi2, matching fit_gauss's convention.
    ok = bin_fit > 0
    pulls = np.where(ok, (counts - bin_fit) / np.sqrt(np.where(ok, bin_fit, 1.0)), 0.0)
    chi2 = float(np.sum((counts[ok] - bin_fit[ok])**2 / bin_fit[ok]))
    ndof = int(ok.sum()) - fit_result.nfit
    return {
        'n_sig': n_sig, 'n_bkg': n_bkg, 'bin_fit': bin_fit, 'pulls': pulls,
        'chi2': chi2, 'ndof': ndof,
        'chi2_per_ndof': chi2 / ndof if ndof > 0 else float('nan'),
    }


def fit_gauss_binned(m12s, m23s, masses, mean_init, sigma_init, c0_init, c1_init,
                     min_candidates=MIN_CANDIDATES):
    """Run a mass fit in every Dalitz bin holding more than `min_candidates`.

    Returns (fit_results, m12_edges, m23_edges), where fit_results maps
    (i, j) -> {'fit', 'n_cand', 'n_events', 'm12_range', 'm23_range'}. Bins with
    too few candidates are simply absent from the dict.
    """
    m12_edges, m23_edges, n = dalitz_edges()
    fit_results = {}
    for i in range(n):
        for j in range(n):
            mask = dalitz_bin_mask(m12s, m23s, m12_edges, m23_edges, i, j)
            masses_in_bin = masses[mask]
            if len(masses_in_bin) <= min_candidates: continue
            fit_result, n_cand = fit_gauss_per_bin(masses_in_bin, mean_init,
                                                   sigma_init, c0_init, c1_init)
            fit_results[(i, j)] = {
                'fit': fit_result,
                'n_cand': n_cand,   # n_cand = n_sig + n_bkg (fixed, not fit)
                'n_events': len(masses_in_bin),
                'm12_range': (m12_edges[i], m12_edges[i + 1]),
                'm23_range': (m23_edges[j], m23_edges[j + 1]),
            }
    return fit_results, m12_edges, m23_edges

In [ ]:
fit_results, m12_edges, m23_edges = fit_gauss_binned(
    m12s, m23s, masses, mean_init, sigma_init, c0_init, c1_init)

n_per_dim = len(m12_edges) - 1
# Signal and background yields per Dalitz bin. A bin that was never fit, or
# whose fit failed, stays 0 -- see the outcome check below for telling those
# two cases apart.
signal_counts = np.zeros((n_per_dim, n_per_dim))
bkg_counts = np.zeros((n_per_dim, n_per_dim))
for (i, j), bin_result in fit_results.items():
    if not bin_result['fit'].valid: continue
    n_sig = bin_result['fit'].values['n_sig']
    signal_counts[i, j] = n_sig
    bkg_counts[i, j] = bin_result['n_cand'] - n_sig

## Fit sanity checks

Three ways a per-bin yield map can lie, and how to tell them apart.

In [ ]:
# --- Sanity check A: which bins actually produced a yield? -------------------
# Hole: signal/background counts equal 0. That happens either because there were
# too few candidates to attempt a fit, or because the fit ran and failed to
# converge (m.valid == False). One way to tell the difference by eye is whether
# the hole is isolated (case a) or surrounded by bins with plenty of candidates
# (case b).
skipped, invalid, valid = [], [], []
for i in range(n_per_dim):
    for j in range(n_per_dim):
        if (i, j) not in fit_results:
            skipped.append((i, j))
        elif not fit_results[(i, j)]['fit'].valid:
            invalid.append((i, j))
        else:
            valid.append((i, j))

print(f'--- Per-bin fit outcome ({n_per_dim}x{n_per_dim} = '
      f'{n_per_dim**2} Dalitz bins) ---')
print(f'  skipped (<= {MIN_CANDIDATES} raw candidates): {len(skipped)}')
print(f'  fit invalid (hole):                {len(invalid)}')
print(f'  fit valid:                         {len(valid)}')

if invalid:
    print("\n--- Raw candidate counts at 'invalid fit' holes ---")
    print('(large n_cand means the hole is a fit-convergence failure)')
    for (i, j) in invalid:
        print(f"  bin ({i:2d},{j:2d}): n_cand={fit_results[(i, j)]['n_cand']:8.0f}")

In [ ]:
# --- Sanity check B: is the fitted background shape physical? ----------------
# c0/c1 float per Dalitz bin (only mean and sigma are fixed in fit_gauss_per_bin).
# The 2nd-order Chebyshev background is NOT guaranteed non-negative over
# [XMIN, XMAX] for every (c0, c1) inside its allowed box: c0 near +/-1 together
# with c1 near -/+1.4 pushes the curve below zero at one edge of the window. A
# per-bin fit can land there and still report valid == True, so this catches
# "quietly wrong" bins that the holes-only check above would miss -- they don't
# show up as 0 in the yield map, just as a bogus but nonzero yield.
x_check = np.linspace(XMIN, XMAX, 200)
neg_bkg_bins = []
for (i, j) in valid:
    r = fit_results[(i, j)]['fit']
    pdf = fit_utils.cheb_bkg_pdf(x_check, XMIN, XMAX, r.values['c0'], r.values['c1'])
    if np.any(pdf < 0): neg_bkg_bins.append((i, j))

print('--- Background positivity check (valid fits only) ---')
print(f'{len(neg_bkg_bins)} / {len(valid)} valid fits have a background curve '
      f'that goes negative somewhere in [XMIN, XMAX].')
if neg_bkg_bins:
    print('First few offending bins (i, j, c0, c1):')
    for (i, j) in neg_bkg_bins[:10]:
        r = fit_results[(i, j)]['fit']
        print(f"  ({i:2d},{j:2d}): c0={r.values['c0']:.3f}  c1={r.values['c1']:.3f}")

In [ ]:
# --- Sanity check C: is raw candidate density smooth around each hole? -------
# If a hole bin's raw n_cand is comparable to its immediate neighbors', the 0
# written into signal_counts/bkg_counts is purely a fit-convergence artifact,
# not a real gap in the underlying Dalitz distribution.
raw = np.full((n_per_dim, n_per_dim), np.nan)
for (i, j), r in fit_results.items():
    raw[i, j] = r['n_cand']

print('--- Neighbor comparison for holes (raw n_cand) ---')
if not invalid:
    print('No holes from non-convergence this run.')
for (i, j) in invalid:
    neighbors = {(di, dj): raw[i + di, j + dj]
                 for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]
                 if 0 <= i + di < n_per_dim and 0 <= j + dj < n_per_dim
                 and not np.isnan(raw[i + di, j + dj])}
    if neighbors:
        neighbor_str = ', '.join(f'{k}: {v:.0f}' for k, v in neighbors.items())
        print(f'  hole ({i:2d},{j:2d}): n_cand={raw[i, j]:.0f}  '
              f'neighbors: {neighbor_str}')

## Fitted yield maps

The Dalitz plot the fits were for: signal and background yields per bin, rather
than raw candidate counts.

In [ ]:
dalitz_font()
plot_dalitz_map(signal_counts, m12_edges, m23_edges, 'Number of signal events',
                r'$\eta \to \mu\mu\gamma$ Dalitz plot (bin-by-bin fitted signal)',
                os.path.join(OUTDIR, 'dalitz_signal_yield.png'))
plot_dalitz_map(bkg_counts, m12_edges, m23_edges, 'Number of background events',
                r'$\eta \to \mu\mu\gamma$ Dalitz plot (bin-by-bin fitted background)',
                os.path.join(OUTDIR, 'dalitz_bkg_yield.png'))

## Results file

The global fit dict from `fit_gauss()` plus a `dalitz` section recording the
per-bin outcome, in the same spirit as `results_gauss.json`: enough to audit the
numbers without the notebook.

In [ ]:
bins = []
for (i, j) in sorted(fit_results):
    bin_result = fit_results[(i, j)]
    fit_result = bin_result['fit']
    entry = {
        'i': i, 'j': j,
        'm12_range': [float(v) for v in bin_result['m12_range']],
        'm23_range': [float(v) for v in bin_result['m23_range']],
        'n_cand': float(bin_result['n_cand']),
        'n_events': int(bin_result['n_events']),
        'valid': bool(fit_result.valid),
    }
    if fit_result.valid:
        # Recount from the same mask the fit used, so chi2 here is the chi2 of
        # the histogram that was fit.
        mask = dalitz_bin_mask(m12s, m23s, m12_edges, m23_edges, i, j)
        counts, _ = np.histogram(masses[mask], bins=MASS_EDGES)
        pred = bin_prediction(fit_result, bin_result['n_cand'], counts.astype(float))
        entry.update({
            'n_sig': round(pred['n_sig'], 2),
            'n_sig_err': round(float(fit_result.errors['n_sig']), 2),
            # derived: n_cand - n_sig (not a free fit parameter)
            'n_bkg': round(pred['n_bkg'], 2),
            'c0': round(float(fit_result.values['c0']), 6),
            'c1': round(float(fit_result.values['c1']), 6),
            'chi2': round(pred['chi2'], 4),
            'ndof': pred['ndof'],
            'chi2_per_ndof': round(pred['chi2_per_ndof'], 6),
            'bkg_goes_negative': (i, j) in neg_bkg_bins,
        })
    bins.append(entry)

result['dalitz'] = {
    'm12_edges': m12_edges.tolist(),
    'm23_edges': m23_edges.tolist(),
    'nbins_total': DALITZ_NBINS,
    'nbins_per_dim': n_per_dim,
    'mass_window': {'xmin': XMIN, 'xmax': XMAX, 'nbins': MASS_NBINS},
    'min_candidates': MIN_CANDIDATES,
    'seeded_from_global_fit': ['mean', 'sigma', 'c0', 'c1'],
    'fixed_in_per_bin_fit': ['mean', 'sigma'],
    'events_dropped_unphysical_m2': n_unphysical,
    'outcome': {'skipped': len(skipped), 'invalid': len(invalid),
                'valid': len(valid)},
    'bins': bins,
}

json_path = os.path.join(OUTDIR, 'dalitz_fit_results.json')
with open(json_path, 'w') as f:
    json.dump(result, f, indent=2)
print(f'[DONE] JSON saved to: {json_path}')

## Per-bin mass spectra

One page per Dalitz bin, so every yield in the maps above can be traced back to
the fit that produced it.

In [ ]:
# Every bin in the grid gets a page -- including the ones fit_gauss_binned
# skipped and the ones whose fit didn't converge. A missing fit says nothing
# about the data, and a page showing the data with no curve on it is exactly
# what's needed to tell "this bin is empty" apart from "this bin has data the
# fit couldn't handle". So the loop runs over the grid itself and looks each bin
# up with .get(), rather than iterating fit_results (which only ever holds the
# bins that were fit).

def plot_bin_page(i, j, counts, n_in_bin, bin_result):
    """One page: the mass spectrum of Dalitz bin (i, j), with its fit and pulls."""
    fit_result = bin_result['fit'] if bin_result is not None else None
    has_fit = fit_result is not None and fit_result.valid

    # Fixed two-panel geometry on every page (spectrum + pulls) so the pages stay
    # comparable when flipping through a few hundred of them.
    fig, (ax, ax_pull) = plt.subplots(
        2, 1, figsize=(8, 6.5), sharex=True,
        gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.07})

    # Data as points with Poisson error bars, not a filled histogram: the whole
    # point of these pages is judging how well a curve tracks the data given the
    # per-bin statistics.
    ax.errorbar(MASS_CENTERS, counts, yerr=np.sqrt(counts), fmt='o', ms=2.5,
                lw=1, color='black', label='Data', zorder=3)

    if has_fit:
        p = fit_result.values
        pred = bin_prediction(fit_result, bin_result['n_cand'], counts)
        # Smooth curves for display: unit-normalized PDFs scaled by their yield
        # and the bin width to land on the counts scale.
        x_fit = np.linspace(XMIN, XMAX, 400)
        y_sig = pred['n_sig'] * MASS_BIN_WIDTH * fit_utils.gauss_pdf(
            x_fit, p['mean'], p['sigma'], xmin=XMIN, xmax=XMAX)
        y_bkg = pred['n_bkg'] * MASS_BIN_WIDTH * fit_utils.cheb_bkg_pdf(
            x_fit, XMIN, XMAX, p['c0'], p['c1'])
        ax.plot(x_fit, y_sig + y_bkg, color='red', lw=1.5, label='Fit', zorder=2)
        ax.plot(x_fit, y_bkg, color='green', ls='--', lw=1.2, label='Background',
                zorder=2)

        ax_pull.bar(MASS_CENTERS, pred['pulls'], width=MASS_BIN_WIDTH,
                    color='red', edgecolor='none')
        for y in (-2, 2): ax_pull.axhline(y, color='gray', ls=':', lw=0.8)

        status = (f"n_cand = {bin_result['n_cand']:.0f}\n"
                  f"n_sig  = {pred['n_sig']:.1f} $\\pm$ "
                  f"{fit_result.errors['n_sig']:.1f}\n"
                  f"n_bkg  = {pred['n_bkg']:.1f}\n"
                  f"c0 = {p['c0']:.3f}   c1 = {p['c1']:.3f}\n"
                  f"$\\chi^2$/ndf = {pred['chi2_per_ndof']:.2f}")
        box_color = 'black'
    else:
        # No curve to draw: say why, and leave the pull panel blank rather than
        # faking residuals against a fit that doesn't exist.
        if bin_result is None:
            status = (f'NO FIT ATTEMPTED\n{n_in_bin} candidates '
                      f'(threshold is {MIN_CANDIDATES})')
        else:
            status = (f"FIT DID NOT CONVERGE\n"
                      f"n_cand = {bin_result['n_cand']:.0f}")
        ax_pull.text(0.5, 0.5, 'no fit', transform=ax_pull.transAxes,
                     ha='center', va='center', color='gray', fontsize=9)
        box_color = 'red'

    ax_pull.set_ylim(-5, 5)
    ax.text(0.02, 0.97, status, transform=ax.transAxes, va='top', ha='left',
            fontsize=8, family='monospace', color=box_color,
            bbox=dict(boxstyle='round', fc='white', ec=box_color, alpha=0.8))
    ax.set_title(f'Bin ({i}, {j}): '
                 f'm12=[{m12_edges[i]:.4f}, {m12_edges[i + 1]:.4f}], '
                 f'm23=[{m23_edges[j]:.4f}, {m23_edges[j + 1]:.4f}]')
    ax.set_ylabel(f'Counts / {MASS_BIN_WIDTH:.2f} MeV')
    ax.set_xlim(XMIN, XMAX)
    ax.legend(loc='upper right', fontsize=8)
    ax_pull.set_xlabel('Mass [MeV]')
    ax_pull.set_ylabel('Pull')
    return fig, has_fit

In [ ]:
# PdfPages is the file, so it is opened ONCE outside the loop -- constructing it
# per iteration reopens and truncates the same path, leaving a 1-page PDF. The
# `with` block also writes the page tree even if a bin raises partway through.
dalitz_font()
pdf_path = os.path.join(OUTDIR, 'per_bin_fits.pdf')
n_pages = n_fitted = 0
with PdfPages(pdf_path) as pdf:
    for i in range(n_per_dim):
        for j in range(n_per_dim):
            # Same mask the fit used, so the page shows the same candidates.
            mask = dalitz_bin_mask(m12s, m23s, m12_edges, m23_edges, i, j)
            counts, _ = np.histogram(masses[mask], bins=MASS_EDGES)
            # .get() -> None for bins that were never fit at all.
            fig, has_fit = plot_bin_page(i, j, counts.astype(float),
                                         int(mask.sum()), fit_results.get((i, j)))
            # One page per bin, then close so a few hundred figures don't all
            # stay resident in memory.
            pdf.savefig(fig)
            plt.close(fig)
            n_pages += 1
            n_fitted += int(has_fit)

print(f'Wrote {n_pages} pages to {pdf_path} '
      f'({n_fitted} with a fit, {n_pages - n_fitted} data-only)')